# mdlARC looped model: Stage 2 full-data d768 x6 checkpoint (180 epochs), submission

Checkpoint: `vast8h200-loopv3-full-d768x6ds-e180-r080` `pretrain/tiny.pt` (8 blocks d768, blocks [2,6) looped x6,
no task embeddings). Loop settings are read from the checkpoint.

Recipe (val40 sweep, 31 runs): the tasks are split into two halves of equal estimated work;
GPUs 0 and 2 take half A and GPUs 1 and 3 half B (`GPU_TASK_GROUPS`). On **each of the four L4s
independently** (no DDP), a full fine-tune of every block on its half's demos, 90 epochs (fewer
if the fine-tune ETA guard refits them), effective batch 40, NorMuon 0.0008 / AdamW 0.00014,
80 training augmentations (context palette); then a leave-one-out decode of its half with 80
augmentations. Each GPU uses its own seed. Vote counts of every candidate grid are summed
over the two GPUs of each half (input copies excluded) and the top two become the attempts.

Inputs: the competition data, the PRIVATE Kaggle Model holding the checkpoint, and the
FlashAttention wheel dataset `mithilavakde/flash-attn-cp313-torch211-cu128`
(no val40 dataset).
Settings: GPU **L4 x4**, Internet **off**.

Safety: an identity `submission.json` is written first and replaced only by validated
predictions; every stage is caught. Training sequences hold at most 4 context demos (an all-30x30
L4 stress test fits 4, not 5 or 6) and retry with 3 after a CUDA OOM; an ETA guard refits the epochs. Each GPU first decodes with 8
augmentations (a cheap answer for every task), then measures that time and runs the full
decode with 80 augmentations, or 40 if 80 would not finish before the decode deadline.
Submit the saved version of this notebook.


In [ ]:
import os, sys
from pathlib import Path
from time import perf_counter
NOTEBOOK_STARTED = perf_counter()
NUM_GPUS = 4
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORKING = Path("/kaggle/working")
WORKING.mkdir(exist_ok=True)
os.chdir(WORKING)
for key in ("MDLARC_CHECKPOINT_QUEUE", "MDLARC_RUN_DIR"):
    os.environ.pop(key, None)
ATTENTION_BACKEND = "flash"
FLASH_ATTN_WHEEL = None
ENABLE_COMPILE = False
os.environ["TORCH_COMPILE_DISABLE"] = "0" if ENABLE_COMPILE else "1"

# ---- Recipe: one independent fine-tune + decode per GPU, votes pooled ----------------
GPU_SEEDS = (42, 1, 2, 3)      # one per GPU; seeds change training order and decode augmentations
GPU_MODELS = (0, 0, 0, 0)      # index into CHECKPOINTS per GPU, e.g. (0, 0, 1, 1) for two models
GPU_TASK_GROUPS = (0, 1, 0, 1) # task half per GPU: GPUs 0+2 take half A, 1+3 half B (each fine-tunes and
                               # decodes only its half); (0, 0, 0, 0) puts every GPU on all tasks
EPOCHS = 90
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 40   # effective batch 40 on each GPU
TRAIN_AUGMENTS = 80
NORMUON_LR = 0.0008
ADAMW_LR = 0.00014
EVAL_AUGMENTS = 80             # full decode per GPU
REDUCED_EVAL_AUGMENTS = 40     # used instead when 80 would miss the decode deadline
QUICK_EVAL_AUGMENTS = 8        # first pass: a cheap answer for every task
FULL_SHARDS_PER_GPU = 4        # a crashed shard loses only its tasks' votes on that GPU
EVAL_BATCH_SIZE = 16
PREFILL_TOKEN_BUDGET = 32768
INFERENCE_SPECULATIVE = "auto"

# ---- Hidden-rerun hardening -----------------------------------------------------------
import json, random, shutil, subprocess, threading, traceback, queue
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
TRAIN_DEADLINE_HOURS = 5.0     # every GPU's fine-tune must end by this notebook-elapsed time
DECODE_DEADLINE_HOURS = 11.3   # decode shards still running then are stopped
DECODE_SAFETY = 1.3            # full-decode estimate = quick-pass seconds x (augments / 8) x this
STARTUP_RESERVE_HOURS = 0.1
TASK_GROUP_COUNT = 1           # becomes the number of GPU_TASK_GROUPS once the tasks are loaded


def task_group(gpu):
    return GPU_TASK_GROUPS[gpu] if TASK_GROUP_COUNT > 1 else 0


MAX_TRAIN_ATTEMPTS = 4
TRAIN_DEMO_CAP = 4             # context demos in training sequences on the first attempt. L4 stress test
                               # (all-30x30 grids): 4 pairs + target = 9,305 tokens fits at 21.6 GiB; 5 and 6 OOM
OOM_DEMO_CAPS = (3,)           # cap after a CUDA OOM (3 pairs + target = 7,444 tokens, 17.5 GiB)
REFIT_TIME_FRACTION = 0.9
LOCAL_CHECKPOINT_DIRS = ("/kaggle/temp", "/tmp")
COPY_THREADS = 16
VALIDATION_FILES = {
    "challenges": {"filename": "arc2_val40_challenges.json", "size": 247515,
                   "sha256": "ce706884dff93678a7295cc69474c7dbe90afddc55044c1bc23b278a7279b0f5"},
    "solutions": {"filename": "arc2_val40_solutions.json", "size": 51494,
                  "sha256": "3af6959c9b527ab0916dd572d20e9377687856cb5adddccdc9fcf19c03b63570"},
}
# Checkpoint identities: size/sha256 None means "exactly one tiny.pt attached; log its SHA".
# required_config is checked after loading. epoch is enforced for an unpinned identity (the dev
# and Stage 2 checkpoints share one config) and only reported for a pinned one.
CHECKPOINTS = [{'label': 'loopv3-full-d768x6ds-e180', 'epoch': 180, 'size': 701113379, 'sha256': None, 'required_config': {'d_model': 768, 'n_layers': 8, 'task_embeddings': False, 'loop_iterations': 6, 'loop_start_layer': 2, 'loop_end_layer': 6, 'loop_input_injection': True, 'loop_exit_weight': 0.3, 'n_predict': 4, 'mtp_weight': 0.3, 'attention_pattern': 'pair_group'}}]
VALIDATION_RUN = False
SAVED_DEFAULTS = {'seed': 42, 'optimizer': 'normuon', 'normuon_momentum': 0.95, 'normuon_momentum_min': 0.85, 'normuon_warmup_steps': 0, 'normuon_cooldown_steps': 0, 'normuon_beta2': 0.95, 'normuon_bucket_size': 0, 'normuon_qkv_lr_multiplier': 1.0, 'normuon_attention_lr_multiplier': 1.0, 'normuon_fc_in_lr_multiplier': 1.0, 'normuon_fc_out_lr_multiplier': 1.0, 'normuon_example_value_latent_lr_multiplier': 1.0, 'normuon_other_lr_multiplier': 1.0, 'adamw_decay_lr_multiplier': 1.0, 'adamw_attention_lr_multiplier': 1.0, 'adamw_token_embedding_lr_multiplier': 1.0, 'adamw_task_factor_lr_multiplier': 1.0, 'adamw_example_value_latent_lr_multiplier': 1.0, 'adamw_no_decay_lr_multiplier': 1.0, 'weight_decay': 0.1, 'attention_weight_decay': 0.01, 'token_embedding_weight_decay': 0.01, 'task_embedding_weight_decay': 0.01, 'grad_clip': 1.0, 'dropout': 0.1, 'embedding_dropout': 0.1, 'attention_dropout': 0.0, 'warmup_pct': 0.005, 'resume_warmup_pct': 0.0, 'wsd_decay_start_pct': 0.8, 'lr_floor': 0.0001, 'mtp_decay': [0.8, 0.9, 1.0], 'augmentation_sampling_mode': 'context_palette', 'training_demo_schedule': 'all', 'max_training_demos': 6, 'inference_demo_policy': 'leave_one_out', 'inference_temperature': None, 'inference_top_k': None, 'pair_identity_vertices': 0}
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    VALIDATION_RUN = False     # a rerun always predicts the hidden tasks
if VALIDATION_RUN:
    print("VALIDATION_RUN is on: this version scores val40 and must not be submitted.", flush=True)
RUN_MODE = "validation" if VALIDATION_RUN else "submission"
print(f"RUN MODE: {RUN_MODE} | epochs {EPOCHS} | batch {TRAIN_BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS} per GPU "
      f"| eval augments {EVAL_AUGMENTS} per GPU | seeds {GPU_SEEDS} | models {GPU_MODELS} "
      f"| task groups {GPU_TASK_GROUPS} | train demo cap "
      f"{TRAIN_DEMO_CAP} | rerun marker {bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))}", flush=True)
PIPELINE_OK, TRAIN_OK = True, False
FAILURES, RUN_FACTS = [], {}
LOG = queue.Queue()            # worker threads never print; the notebook thread relays


def hours_elapsed():
    return (perf_counter() - NOTEBOOK_STARTED) / 3600


def record_failure(stage, error):
    FAILURES.append({"stage": stage, "error": repr(error), "hours": round(hours_elapsed(), 3)})
    LOG.put(f"STAGE FAILED: {stage} at {hours_elapsed():.2f}h: {error!r}\n{traceback.format_exc()[-3000:]}")


def relay():
    while True:
        try:
            print(LOG.get_nowait(), flush=True)
        except queue.Empty:
            return


def identity_submission(tasks):
    return {key: [{"attempt_1": pair["input"], "attempt_2": pair["input"]} for pair in task["test"]]
            for key, task in tasks.items()}


try:
    names = {"arc-agi_test_challenges.json", "arc-agi_test-challenges.json"}
    found = sorted(p for p in Path("/kaggle/input").rglob("*.json") if p.name in names)
    preferred = [p for p in found if "arc-prize-2026-arc-agi-2" in p.parts]
    if not preferred:
        raise FileNotFoundError(f"No competition test file under /kaggle/input; found {found}")
    COMPETITION_TEST_FILE = preferred[0]
    raw_tasks = json.loads(COMPETITION_TEST_FILE.read_text())
    fallback_tmp = WORKING / "submission.json.tmp"
    fallback_tmp.write_text(json.dumps(identity_submission(raw_tasks)))
    fallback_tmp.replace(WORKING / "submission.json")
    print("Identity fallback submission.json written for", len(raw_tasks), "tasks", flush=True)
    del raw_tasks
except Exception as error:
    record_failure("fallback", error)
relay()


In [ ]:
if PIPELINE_OK:
    try:
        import importlib.util
        import platform
        import hashlib
        import torch

        def restore_wheel_filename(source):
            source = Path(source)
            manifest_path = source.parent / "wheel-manifest.json"
            if not manifest_path.is_file():
                return source
            manifest = json.loads(manifest_path.read_text())
            name = manifest["wheel"]
            if Path(name).name != name or not name.endswith(".whl"):
                raise ValueError("Invalid wheel filename in build manifest.")
            with source.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            if digest != manifest["wheel_sha256"]:
                raise ValueError("FlashAttention wheel checksum failed.")
            target = WORKING / name
            if source.resolve() != target.resolve():
                shutil.copyfile(source, target)
            return target

        print("Python:", sys.version)
        print("Platform:", platform.platform(), platform.machine())
        print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
        if not torch.cuda.is_available():
            raise RuntimeError("Select GPU L4 x 4 in Kaggle Settings, then restart the session.")
        print("Allocated GPU count:", torch.cuda.device_count(), flush=True)
        if torch.cuda.device_count() != NUM_GPUS:
            print(f"WARNING: expected {NUM_GPUS} GPUs, got {torch.cuda.device_count()}; "
                  "running one independent worker per available GPU.", flush=True)
            NUM_GPUS = torch.cuda.device_count()
        if NUM_GPUS < 1 or not torch.cuda.is_bf16_supported():
            raise RuntimeError("This optimizer needs a CUDA GPU with bfloat16 support; choose L4.")
        print("Using:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
        print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
        subprocess.run(["nvidia-smi"], check=True)
        for package in ("numpy", "numba"):
            if importlib.util.find_spec(package) is None:
                raise RuntimeError(f"Missing {package}; attach an offline wheel. Do not enable Internet for submission.")

        IN_FLASH_SETUP = True  # a FlashAttention failure falls back to native attention below
        flash_probe = [sys.executable, "-c", "from flash_attn import flash_attn_varlen_func; import flash_attn; print(flash_attn.__version__)"]
        if ATTENTION_BACKEND == "flash":
            probe = subprocess.run(flash_probe, text=True, capture_output=True)
            if FLASH_ATTN_WHEEL is not None or probe.returncode:
                if FLASH_ATTN_WHEEL is None:
                    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
                    wheels = sorted(p for p in Path("/kaggle/input").rglob("*.whl")
                                    if p.name.startswith("flash_attn-") and tag in p.name)
                    if len(wheels) != 1:
                        raise RuntimeError(
                            f"No working FlashAttention import; found {len(wheels)} wheels for {tag}. "
                            "Attach the flash-attn wheel dataset, or set ATTENTION_BACKEND='native'.\n"
                            + probe.stderr[-3000:])
                    FLASH_ATTN_WHEEL = wheels[0]
                wheel = Path(FLASH_ATTN_WHEEL)
                if not wheel.is_file() or wheel.suffix != ".whl":
                    raise FileNotFoundError(f"Offline wheel not found: {wheel}")
                wheel = restore_wheel_filename(wheel)
                for dependency in sorted(Path(FLASH_ATTN_WHEEL).parent.glob("einops-*.whl")):
                    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                                    "--no-deps", str(dependency)], check=True)
                print("Installing offline wheel:", wheel)
                subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                                "--force-reinstall", str(wheel)], check=True)
                probe = subprocess.run(flash_probe, text=True, capture_output=True)
            if probe.returncode:
                raise RuntimeError("FlashAttention wheel is incompatible with this environment.\n" + probe.stderr[-5000:])
            print("FlashAttention:", probe.stdout.strip())
        else:
            print("Using native PyTorch attention; no FlashAttention wheel will be used.")
        IN_FLASH_SETUP = False
    except Exception as error:
        record_failure("environment", error)
        if globals().get("IN_FLASH_SETUP") and ATTENTION_BACKEND == "flash":
            # Slower than FA2 but far better than an identity submission; the time guards still apply.
            ATTENTION_BACKEND = "native"
            print("FlashAttention unavailable: continuing with native PyTorch attention.", flush=True)
        else:
            PIPELINE_OK = False
relay()


In [ ]:
if PIPELINE_OK:
    try:
        import base64
        import hashlib
        import io
        import zipfile

        RUNTIME_SHA256 = 'c364b2adc74935b10a682d1c5f5bbcbf48e4dfd871be32e23ee40e66f0732bd2'
        RUNTIME_B64 = '...'
        runtime_bytes = base64.b64decode(RUNTIME_B64)
        if hashlib.sha256(runtime_bytes).hexdigest() != RUNTIME_SHA256:
            raise RuntimeError("Embedded runtime checksum failed; re-upload the original notebook.")
        RUNTIME = WORKING / "mdlarc_runtime"
        with zipfile.ZipFile(io.BytesIO(runtime_bytes)) as archive:
            for entry in archive.infolist():
                target = (RUNTIME / entry.filename).resolve()
                target.relative_to(RUNTIME.resolve())
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(entry))
        sys.path.insert(0, str(RUNTIME / "src"))
        sys.path.insert(0, str(RUNTIME / "kaggle"))
        del runtime_bytes, RUNTIME_B64
        if ATTENTION_BACKEND == "native":
            sys.modules["flash_attn"] = None
            sys.modules["flash_attn_interface"] = None
        # Import everything the workers import, so a missing module fails here, visibly.
        import build, evaluate, train, tinytransformer, research_options, next_latent
        import submission_utils as ku
        import multi_gpu as mg
        import loop_pool
        if ATTENTION_BACKEND == "flash" and tinytransformer.flash_attn_varlen_func is None:
            raise RuntimeError("FlashAttention was not imported into the notebook kernel. Restart and Run All.")
        print("Runtime SHA-256:", RUNTIME_SHA256, flush=True)
    except Exception as error:
        PIPELINE_OK = False
        record_failure("runtime extraction", error)
relay()


In [ ]:
if PIPELINE_OK:
    try:
        import gc
        from concurrent.futures import ThreadPoolExecutor, as_completed

        def mount_of(path):
            path, best = os.path.realpath(path), ("/", "unknown")
            try:
                with open("/proc/mounts") as mounts:
                    for line in mounts:
                        fields = line.split()
                        if len(fields) < 3:
                            continue
                        point = fields[1].replace("\\040", " ")
                        inside = path == point or path.startswith(point.rstrip("/") + "/")
                        if inside and len(point) >= len(best[0]):
                            best = (point, fields[2])
            except OSError:
                pass
            return best

        def describe_disks():
            for place in ("/kaggle/input", "/kaggle/working", *LOCAL_CHECKPOINT_DIRS, "/dev/shm"):
                if Path(place).exists():
                    usage = shutil.disk_usage(place)
                    print(f"Disk {place}: {mount_of(place)[1]}, free {usage.free / 2**30:.1f} "
                          f"of {usage.total / 2**30:.1f} GiB", flush=True)
            memory = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
            print(f"RAM: {memory / 2**30:.1f} GiB", flush=True)

        def pick_checkpoint(identity):
            names = {"pretrain__tiny.pt", "tiny.pt", identity.get("filename") or "tiny.pt"}
            found = sorted(p for p in Path("/kaggle/input").rglob("*.pt") if p.name in names)
            if identity.get("size") is not None:
                found = [p for p in found if p.stat().st_size == identity["size"]]
            elif len(found) != 1:
                raise FileNotFoundError(f"Attach exactly one checkpoint for {identity['label']}; found {found}")
            if not found:
                raise FileNotFoundError(f"No checkpoint of the expected size for {identity['label']}")
            found.sort(key=lambda p: ("models" not in p.parts, str(p)))
            return found[0]

        def parallel_copy(source, destination, threads, chunk=32 << 20):
            size = source.stat().st_size
            partial = destination.with_suffix(".part")
            started, copied = perf_counter(), 0
            with source.open("rb") as reader, partial.open("wb") as writer:
                writer.truncate(size)

                def copy_range(offset):
                    length, done = min(chunk, size - offset), 0
                    while done < length:
                        data = os.pread(reader.fileno(), length - done, offset + done)
                        if not data:
                            raise OSError(f"Short read at byte {offset + done}")
                        view = memoryview(data)
                        while view:
                            written = os.pwrite(writer.fileno(), view, offset + done)
                            if written <= 0:
                                raise OSError(f"No write progress at byte {offset + done}")
                            view, done = view[written:], done + written
                    return length

                with ThreadPoolExecutor(threads) as pool:
                    futures = [pool.submit(copy_range, offset) for offset in range(0, size, chunk)]
                    try:
                        for future in as_completed(futures):
                            copied += future.result()
                    except BaseException:
                        for future in futures:
                            future.cancel()
                        raise
            partial.replace(destination)
            return perf_counter() - started

        def local_checkpoint(index, identity):
            source = pick_checkpoint(identity)
            print(f"Checkpoint {index} ({identity['label']}): {source} on {mount_of(source)[1]}", flush=True)
            path = source
            for place in LOCAL_CHECKPOINT_DIRS:
                try:
                    folder = Path(place) / f"mdlarc-checkpoint-{index}"
                    folder.mkdir(parents=True, exist_ok=True)
                    if shutil.disk_usage(folder).free < source.stat().st_size + 4 * 2**30:
                        continue
                    destination = folder / "pretrain__tiny.pt"
                    seconds = parallel_copy(source, destination, COPY_THREADS)
                    print(f"Copied checkpoint {index} to {destination} in {seconds:.0f}s", flush=True)
                    path = destination
                    break
                except Exception as error:
                    print(f"Local copy to {place} failed: {error!r}", flush=True)
            digest = ku.sha256_file(path)
            if identity.get("sha256") is not None and digest != identity["sha256"]:
                if path == source:
                    raise ValueError(f"Checkpoint {index} SHA-256 differs from its identity")
                print("Local copy failed verification; using the mounted file", flush=True)
                path = source
                digest = ku.sha256_file(path)
                if digest != identity["sha256"]:
                    raise ValueError(f"Checkpoint {index} SHA-256 differs from its identity")
            loaded = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
            config = loaded["config"]
            for key, expected in identity["required_config"].items():
                if config.get(key) != expected:
                    raise ValueError(f"Checkpoint {index}: config {key}={config.get(key)!r}, expected {expected!r}")
            if loaded.get("epoch") != identity.get("epoch"):
                message = f"checkpoint {index} epoch {loaded.get('epoch')} (identity says {identity.get('epoch')})"
                if identity.get("sha256") is None:
                    # Without a SHA, the epoch is the only check that tells the dev and Stage 2 files
                    # apart. To submit another epoch on purpose, set CHECKPOINTS[index]["epoch"] in cell 1.
                    raise ValueError(f"{message}: wrong checkpoint attached, or update the CHECKPOINTS epoch")
                print(f"NOTE: {message}", flush=True)
            # The SHA pins the bytes; workers re-check these fields after loading.
            request_identity = {"epoch": loaded.get("epoch"),
                                "augmentation_sampling_mode": loaded["augmentation_sampling_mode"],
                                "config": dict(identity["required_config"])}
            facts = {"label": identity["label"], "path": str(path), "size": path.stat().st_size,
                     "sha256": digest, "epoch": loaded.get("epoch"),
                     "loop": {key: config.get(key) for key in config if key.startswith("loop_")}}
            print(f"Checkpoint {index} verified: {json.dumps(facts)}", flush=True)
            del loaded
            gc.collect()
            return path, request_identity, facts

        describe_disks()
        LOCAL_CHECKPOINTS = {}
        for index in sorted(set(GPU_MODELS[:NUM_GPUS])):
            path, request_identity, facts = local_checkpoint(index, CHECKPOINTS[index])
            LOCAL_CHECKPOINTS[index] = (path, request_identity)
            RUN_FACTS[f"checkpoint_{index}"] = facts
        if VALIDATION_RUN:
            def find_validation_file(kind):
                identity = VALIDATION_FILES[kind]
                matches = sorted(Path("/kaggle/input").rglob(identity["filename"]))
                if len(matches) != 1:
                    raise FileNotFoundError(f"Attach exactly one {identity['filename']}; found {len(matches)}")
                return ku.verify_dataset_file(matches[0], identity)
            TEST_CHALLENGES = find_validation_file("challenges")
            print("VALIDATION RUN: adapting and decoding", TEST_CHALLENGES, flush=True)
        else:
            TEST_CHALLENGES = ku.find_test_challenges(Path("/kaggle/input"))
        data_path = WORKING / "runtime_challenges.json"
        challenges = ku.prepare_challenges(TEST_CHALLENGES, data_path)
        print("Tasks", len(challenges), "demos", sum(len(t["train"]) for t in challenges.values()),
              "queries", sum(len(t["test"]) for t in challenges.values()), f"at {hours_elapsed():.2f}h", flush=True)
        groups_used = sorted(set(GPU_TASK_GROUPS[:NUM_GPUS]))
        if groups_used != list(range(len(groups_used))):
            raise ValueError(f"GPU_TASK_GROUPS {GPU_TASK_GROUPS} must use groups 0..k-1 on the {NUM_GPUS} GPUs")
        TASK_GROUP_COUNT = len(groups_used) if len(challenges) >= len(groups_used) else 1
        GROUP_CHALLENGES, GROUP_PATHS = {0: challenges}, {0: data_path}
        if TASK_GROUP_COUNT > 1:
            def task_cost(task):
                size = lambda grid: len(grid) * (len(grid[0]) + 1)
                demos = [1 + size(pair["input"]) + size(pair["output"]) for pair in task["train"]]
                train = len(demos) * sum(demos)  # each demo is a target with the others as context
                decode = sum(sum(demos) + 1 + 2 * size(pair["input"]) for pair in task["test"])  # output ~ input
                return train, decode

            costs = {task_id: task_cost(task) for task_id, task in challenges.items()}
            totals = [max(1, sum(cost[part] for cost in costs.values())) for part in (0, 1)]
            weight = {task_id: cost[0] / totals[0] + cost[1] / totals[1] for task_id, cost in costs.items()}
            loads, members = [0.0] * TASK_GROUP_COUNT, [[] for _ in range(TASK_GROUP_COUNT)]
            for task_id in sorted(weight, key=lambda key: (-weight[key], key)):  # largest first, lightest group
                group = min(range(TASK_GROUP_COUNT), key=lambda index: (loads[index], index))
                loads[group] += weight[task_id]
                members[group].append(task_id)
            for group, ids in enumerate(members):
                GROUP_CHALLENGES[group] = {task_id: challenges[task_id] for task_id in sorted(ids)}
                GROUP_PATHS[group] = WORKING / f"runtime_challenges_group{group}.json"
                GROUP_PATHS[group].write_text(json.dumps(GROUP_CHALLENGES[group]))
        for group in range(TASK_GROUP_COUNT):
            subset = GROUP_CHALLENGES[group]
            print(f"Task group {group}: GPUs {[gpu for gpu in range(NUM_GPUS) if task_group(gpu) == group]}, "
                  f"tasks {len(subset)} demos {sum(len(t['train']) for t in subset.values())} "
                  f"queries {sum(len(t['test']) for t in subset.values())}", flush=True)
        RUN_FACTS["task_groups"] = {group: sorted(GROUP_CHALLENGES[group]) for group in range(TASK_GROUP_COUNT)}
    except Exception as error:
        PIPELINE_OK = False
        record_failure("checkpoint and challenges", error)
relay()


In [ ]:
STATE = {gpu: {"seed": GPU_SEEDS[gpu % len(GPU_SEEDS)], "model": GPU_MODELS[gpu], "task_group": task_group(gpu),
               "attempts": [],
               "train_ok": False, "passes": {}} for gpu in range(NUM_GPUS)}
STATE_LOCK = threading.Lock()


def request_for(gpu, tag, epochs, eta_hours, demo_cap):
    run_dir = WORKING / "runs" / f"g{gpu}_{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint, identity = LOCAL_CHECKPOINTS[GPU_MODELS[gpu]]
    defaults = dict(SAVED_DEFAULTS, seed=STATE[gpu]["seed"], max_training_demos=demo_cap)
    request = {
        "working": str(WORKING), "run_dir": str(run_dir), "num_gpus": 1,
        "checkpoint_path": str(checkpoint), "checkpoint_identity": identity,
        "data_path": str(GROUP_PATHS[task_group(gpu)]), "save_path": str(run_dir / "finetuned.pt"),
        "inference_checkpoint": str(run_dir / "inference.pt"), "defaults": defaults,
        "attention_backend": ATTENTION_BACKEND, "enable_compile": ENABLE_COMPILE,
        "train_log_mode": "epoch", "smoke_test": False, "max_finetune_hours": eta_hours,
        "eval_augments": EVAL_AUGMENTS, "eval_batch_size": EVAL_BATCH_SIZE,
        "prefill_token_budget": PREFILL_TOKEN_BUDGET, "inference_speculative": INFERENCE_SPECULATIVE,
        "finetune": {"epochs": epochs, "batch_size": TRAIN_BATCH_SIZE,
                     "accumulation": GRADIENT_ACCUMULATION_STEPS, "max_augments": TRAIN_AUGMENTS,
                     "num_workers": 0, "normuon_lr": NORMUON_LR, "adamw_lr": ADAMW_LR},
    }
    path = run_dir / "request.json"
    mg.write_json(path, request)
    return run_dir, request, path


def run_logged(command, gpu, log_path, timeout):
    env = mg.worker_environment(gpu=gpu)
    env["LOCAL_RANK"] = "0"  # one independent process per GPU, no torchrun: its device is cuda:0
    with log_path.open("w") as log:
        process = subprocess.Popen(command, env=env, stdout=log,
                                   stderr=subprocess.STDOUT, text=True)
        try:
            return process.wait(timeout=max(1, timeout))
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            return "killed (deadline)"


def classify(text):
    # torch.OutOfMemoryError ("CUDA out of memory") and driver/cuBLAS/cuDNN allocation failures
    # ("CUDA error: out of memory", "CUBLAS_STATUS_ALLOC_FAILED") all call for the same smaller retry.
    lowered = text.lower()
    if "outofmemoryerror" in lowered or "out of memory" in lowered or "alloc_failed" in lowered:
        return "oom"
    if "ETA exceeds" in text:
        return "eta"
    return "other"


def epoch_seconds(text):
    import re
    return [float(value) for value in re.findall(r"Epoch \d+/\d+ compute completed in ([0-9.]+)s", text)]


def train_gpu(gpu):
    """Fine-tune on one GPU with the OOM ladder and ETA refit; returns its inference checkpoint."""
    epochs, demo_cap, other_retry = EPOCHS, TRAIN_DEMO_CAP, True
    for attempt in range(MAX_TRAIN_ATTEMPTS):
        budget = TRAIN_DEADLINE_HOURS - hours_elapsed() - STARTUP_RESERVE_HOURS
        tag = f"a{attempt}_e{epochs}_d{demo_cap}"
        if epochs < min(10, EPOCHS) or budget <= 0:
            with STATE_LOCK:
                FAILURES.append({"stage": f"adaptation gpu{gpu} budget", "kind": "eta", "hours": round(hours_elapsed(), 3),
                                 "error": f"{budget:.2f}h left for {epochs} epochs"})
            LOG.put(f"[gpu{gpu}] too little adaptation time ({budget:.2f}h for {epochs} epochs)")
            return None
        run_dir, request, path = request_for(gpu, tag, epochs, budget, demo_cap)
        LOG.put(f"[gpu{gpu}] ADAPTATION {tag}: seed {STATE[gpu]['seed']}, model {GPU_MODELS[gpu]}, "
                f"ETA limit {budget:.2f}h, start {hours_elapsed():.2f}h")
        started = perf_counter()
        log_path = run_dir / "train.log"
        command = [sys.executable, "-u", str(Path(mg.__file__)), "--phase", "train", "--request", str(path)]
        status = run_logged(command, gpu, log_path, (TRAIN_DEADLINE_HOURS - hours_elapsed()) * 3600)
        text = log_path.read_text(errors="replace") if log_path.is_file() else ""
        seconds = epoch_seconds(text)
        try:
            if status != 0:
                raise RuntimeError(f"training exited with {status!r}")
            report = json.loads((run_dir / "train-0.json").read_text())
            if report["epochs"] != epochs or report["optimizer_updates"] < 1:
                raise RuntimeError("training report failed verification")
            mg.make_inference_checkpoint(Path(request["save_path"]), Path(request["inference_checkpoint"]), epochs)
            Path(request["save_path"]).unlink(missing_ok=True)
            row = {"tag": tag, "result": "ok", "seconds": round(perf_counter() - started, 1),
                   "epoch_seconds": [round(s, 1) for s in seconds], "updates": report["optimizer_updates"],
                   "peak_gib": round(report.get("peak_allocated_gib", 0.0), 2)}
            with STATE_LOCK:
                STATE[gpu]["attempts"].append(row)
                STATE[gpu]["train_ok"] = True
            LOG.put(f"[gpu{gpu}] ADAPTATION COMPLETE {tag} at {hours_elapsed():.2f}h: {json.dumps(row)}")
            return request
        except Exception as error:
            kind = classify(text) if status != 0 else "other"
            for stale in run_dir.glob("*.pt"):
                stale.unlink(missing_ok=True)
            with STATE_LOCK:
                STATE[gpu]["attempts"].append({"tag": tag, "result": kind, "epoch_seconds": seconds})
                FAILURES.append({"stage": f"adaptation gpu{gpu} {tag}", "error": repr(error),
                                 "kind": kind, "hours": round(hours_elapsed(), 3)})
            LOG.put(f"[gpu{gpu}] adaptation {tag} failed ({kind}): {error!r}; epoch seconds {seconds}\n"
                    + text[-2500:])
        if kind == "oom":
            smaller = [cap for cap in OOM_DEMO_CAPS if cap < demo_cap]
            if not smaller:
                return None
            demo_cap = smaller[0]
        elif kind == "other":
            if not other_retry:
                return None
            other_retry = False  # one bounded retry for an unclassified failure
        if seconds and kind in ("eta", "other"):
            first = seconds[0]
            warm = sum(seconds[1:]) / (len(seconds) - 1) if len(seconds) > 1 else first
            usable = REFIT_TIME_FRACTION * (TRAIN_DEADLINE_HOURS - hours_elapsed() - STARTUP_RESERVE_HOURS) * 3600 - 120
            fitted = int((usable - first) / warm) + 1 if usable > first else 0
            epochs = min(epochs - 1 if kind == "eta" else epochs, fitted)
    return None


def decode_gpu(gpu, request, tag, augments, shards):
    """Decode every task on one GPU in `shards` sequential pieces; an OOM shard retries smaller."""
    pass_dir = WORKING / "runs" / f"g{gpu}_{tag}"
    pass_dir.mkdir(parents=True, exist_ok=True)
    for stale in [*pass_dir.glob("shard-*.json"), *pass_dir.glob("decode-*.json")]:
        stale.unlink()
    base = dict(request, run_dir=str(pass_dir), eval_augments=augments)
    started, done = perf_counter(), []
    for shard in range(shards):
        for retry, scale in enumerate((1, 4)):
            remaining = DECODE_DEADLINE_HOURS * 3600 - (perf_counter() - NOTEBOOK_STARTED)
            if remaining < 120:
                LOG.put(f"[gpu{gpu}] {tag} shard {shard + 1}/{shards}: skipped (deadline)")
                break
            path = pass_dir / f"request-{retry}.json"
            mg.write_json(path, dict(base, eval_batch_size=max(1, EVAL_BATCH_SIZE // scale),
                                     prefill_token_budget=max(4096, PREFILL_TOKEN_BUDGET // scale)))
            log_path = pass_dir / f"worker-{shard}-{retry}.log"
            status = run_logged(mg.inference_command(path, shard, shards), gpu, log_path, remaining)
            ok = (pass_dir / f"shard-{shard}.json").is_file()
            LOG.put(f"[gpu{gpu}] {tag} {augments} augs shard {shard + 1}/{shards}: exit {status!r}, "
                    f"answers {'yes' if ok else 'no'} at {hours_elapsed():.2f}h")
            if ok:
                done.append(shard)
                break
            text = log_path.read_text(errors="replace") if log_path.is_file() else ""
            LOG.put(text[-2500:])
            if classify(text) != "oom":
                break
    seconds = perf_counter() - started
    with STATE_LOCK:
        STATE[gpu]["passes"][tag] = {"augments": augments, "shards": shards, "complete": done,
                                     "seconds": round(seconds, 1), "dir": str(pass_dir)}
    return seconds, len(done) == shards


def gpu_worker(gpu):
    try:
        request = train_gpu(gpu)
        if request is None:
            LOG.put(f"[gpu{gpu}] no adaptation succeeded; this GPU contributes no votes")
            return
        quick_seconds, quick_ok = decode_gpu(gpu, request, "quick", QUICK_EVAL_AUGMENTS, 1)
        remaining = DECODE_DEADLINE_HOURS * 3600 - (perf_counter() - NOTEBOOK_STARTED)
        chosen = None
        if quick_ok:
            for augments in (EVAL_AUGMENTS, REDUCED_EVAL_AUGMENTS):
                if quick_seconds * augments / QUICK_EVAL_AUGMENTS * DECODE_SAFETY < remaining:
                    chosen = augments
                    break
        LOG.put(f"[gpu{gpu}] quick decode {quick_seconds:.0f}s; {remaining / 3600:.2f}h left; "
                f"full decode with {chosen} augmentations")
        with STATE_LOCK:
            STATE[gpu]["full_augments"] = chosen
        if chosen:
            decode_gpu(gpu, request, "full", chosen, FULL_SHARDS_PER_GPU)
    except Exception as error:
        record_failure(f"gpu{gpu} worker", error)


def gpu_votes(gpu):
    """This GPU's candidates per test output: full-pass dumps first, quick pass for the rest."""
    votes = {}
    with STATE_LOCK:  # workers insert passes concurrently
        passes = {tag: dict(info) for tag, info in STATE[gpu]["passes"].items()}
    for tag in ("quick", "full"):
        info = passes.get(tag)
        if not info:
            continue
        pass_dir = Path(info["dir"])
        ids = mg.task_shards(GROUP_CHALLENGES[task_group(gpu)], info["shards"])  # this GPU decodes its group only
        for shard in info["complete"]:
            dump = WORKING / "runs" / pass_dir.name / f"decode-{shard}" / "candidates.json.gz"
            try:
                part = loop_pool.read_dump(dump)
            except Exception:
                part = loop_pool.from_submission(json.loads((pass_dir / f"shard-{shard}.json").read_text()),
                                                 challenges)
            for key in [k for k in part if k[0] in ids[shard]]:
                votes[key] = part[key]  # a later (full) pass replaces the quick one
    return votes


def write_pooled(label):
    sources = [gpu_votes(gpu) for gpu in range(NUM_GPUS)]
    pooled = loop_pool.pool(sources)
    predictions = loop_pool.submission(pooled, challenges)
    ku.validate_submission(predictions, challenges)
    ku.validate_submission(predictions, json.loads(Path(TEST_CHALLENGES).read_text()))
    output = WORKING / ("validation_predictions.json" if VALIDATION_RUN else "submission.json")
    mg.write_json(output, predictions)
    answered = sum(any(key[0] == task for key in pooled) for task in challenges)
    LOG.put(f"{output.name} updated ({label}): pooled votes from "
            f"{sum(bool(s) for s in sources)} GPUs; model candidates for {answered}/{len(challenges)} tasks")
    return pooled, sources, answered


PROGRESS_SECONDS = 10       # check each GPU's fine-tune log this often; a line prints for every new epoch


def progress_lines(seen):
    """Latest epoch line of each GPU's running fine-tune log (the worker reads it only at the end)."""
    import re
    lines = []
    for gpu in range(NUM_GPUS):
        logs = sorted((WORKING / "runs").glob(f"g{gpu}_*/train.log"), key=lambda path: path.stat().st_mtime)
        if not logs:
            continue
        with logs[-1].open("rb") as handle:
            handle.seek(0, 2)
            handle.seek(max(0, handle.tell() - 16384))
            tail = handle.read().decode(errors="replace")
        epochs = re.findall(r"Epoch (\d+)/(\d+) compute completed in ([0-9.]+)s", tail)
        eta = re.findall(r"Estimated DDP adaptation: ([0-9.]+) hours", tail)
        run, printed = logs[-1].parent.name, seen.get(gpu, ("", 0))
        for done, total, last in epochs:  # every epoch not yet printed (a 10 s check can span two)
            if (run, int(done)) <= (printed[0], printed[1]) and run == printed[0]:
                continue
            lines.append(f"[gpu{gpu}] fine-tune {run.split('_', 1)[1]}: epoch {done}/{total} took {float(last):.1f} s"
                         + (f"; projected total fine-tune {float(eta[-1]):.2f} h" if eta and int(done) >= 3 else "")
                         + f" (notebook at {hours_elapsed():.2f}h)")
            seen[gpu] = (run, int(done))
    return lines


MODEL_TASKS, POOLED, SOURCES = 0, {}, []
if PIPELINE_OK:
    threads = [threading.Thread(target=gpu_worker, args=(gpu,), daemon=True) for gpu in range(NUM_GPUS)]
    for thread in threads:
        thread.start()
    written, progress_seen, next_progress = set(), {}, perf_counter()
    while any(thread.is_alive() for thread in threads):
        relay()
        if perf_counter() >= next_progress:
            next_progress = perf_counter() + PROGRESS_SECONDS
            try:
                for line in progress_lines(progress_seen):
                    print(line, flush=True)
            except Exception as error:
                print(f"progress read failed: {error!r}", flush=True)
        try:  # an exception here would end Run All before the final cell
            with STATE_LOCK:
                snapshot = {(gpu, tag) for gpu in range(NUM_GPUS) for tag in STATE[gpu]["passes"]}
            if snapshot - written:
                written = snapshot
                POOLED, SOURCES, MODEL_TASKS = write_pooled(f"{len(snapshot)} decode passes so far")
        except Exception as error:
            record_failure("pooled write", error)
        for thread in threads:  # wait on a live worker (at most 10 s); joining a finished one would spin
            if thread.is_alive():
                thread.join(timeout=10)
                break
    relay()
    TRAIN_OK = any(STATE[gpu]["train_ok"] for gpu in range(NUM_GPUS))
    try:
        POOLED, SOURCES, MODEL_TASKS = write_pooled("final")
    except Exception as error:
        record_failure("pooled write", error)
    relay()
    RUN_FACTS["gpus"] = STATE
    try:
        print("PER-GPU SUMMARY:", json.dumps(STATE, indent=1, default=str), flush=True)
    except Exception as error:
        print("Per-GPU summary failed:", repr(error), flush=True)


In [ ]:
def _valid_grid(grid):
    return (isinstance(grid, list) and 1 <= len(grid) <= 30 and isinstance(grid[0], list) and 1 <= len(grid[0]) <= 30
            and all(isinstance(row, list) and len(row) == len(grid[0])
                    and all(type(color) is int and 0 <= color <= 9 for color in row) for row in grid))


def _validate_submission(submission, challenges):
    """submission_utils.validate_submission, repeated here so the check runs even if the runtime never loaded."""
    if not isinstance(submission, dict) or set(submission) != set(challenges):
        raise ValueError("Submission task ids must exactly match the challenge task ids.")
    for task_id, task in challenges.items():
        guesses = submission[task_id]
        if not isinstance(guesses, list) or len(guesses) != len(task["test"]):
            raise ValueError(f"{task_id}: wrong number of test predictions")
        for index, guess in enumerate(guesses):
            if not isinstance(guess, dict) or set(guess) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{task_id}/{index}: expected exactly two attempts")
            for key in ("attempt_1", "attempt_2"):
                if not _valid_grid(guess[key]):
                    raise ValueError(f"{task_id}/{index}/{key}: invalid output grid")


try:
    for checkpoint in (WORKING / "runs").rglob("*.pt"):
        checkpoint.unlink()
    for place in LOCAL_CHECKPOINT_DIRS:
        for folder in Path(place).glob("mdlarc-checkpoint-*"):
            shutil.rmtree(folder, ignore_errors=True)
except Exception as error:
    print("Checkpoint cleanup failed:", repr(error), flush=True)
VALIDATION_SCORE = None
if VALIDATION_RUN:
    try:
        output = WORKING / "validation_predictions.json"
        if not output.is_file():
            print("VALIDATION: no model predictions were written; see the failures below.", flush=True)
        else:
            solutions = json.loads(find_validation_file("solutions").read_text())
            pooled_score = ku.score_validation(json.loads(output.read_text()), challenges, solutions)
            per_task = pooled_score.pop("per_task")
            VALIDATION_SCORE = {"pooled_top2": pooled_score["top2"], "pooled_top1": pooled_score["top1"],
                                "pooled_oracle": loop_pool.oracle_score(POOLED, challenges, solutions),
                                "pooled_oracle_with_input_copies": loop_pool.oracle_score(
                                    POOLED, challenges, solutions, include_input_copies=True),
                                "per_gpu": {}}
            for gpu, votes in enumerate(SOURCES):
                if votes:  # a single GPU is scored on its own task group only
                    mine = GROUP_CHALLENGES.get(task_group(gpu), challenges)
                    answers = {task_id: solutions[task_id] for task_id in mine}
                    single = loop_pool.pool([votes])
                    top = ku.score_validation(loop_pool.submission(single, mine), mine, answers)
                    VALIDATION_SCORE["per_gpu"][gpu] = {
                        "task_group": task_group(gpu), "tasks": len(mine),
                        "top2": round(top["top2"]["percent"], 2),
                        "oracle": round(loop_pool.oracle_score(single, mine, answers)["percent"], 2),
                        "full_augments": STATE[gpu].get("full_augments")}
            print("VALIDATION SCORE:", json.dumps(VALIDATION_SCORE, indent=2), flush=True)
            print(f"VAL40 RESULT: pooled top-2 {pooled_score['top2']['percent']:.2f} | pooled oracle "
                  f"{VALIDATION_SCORE['pooled_oracle']['percent']:.2f} (600M bar 32.08 / 42.08)", flush=True)
            print("Tasks solved within two attempts:",
                  sorted(key for key, row in per_task.items() if row["top2_correct"]), flush=True)
    except Exception as error:
        record_failure("validation scoring", error)
relay()
try:
    info = {"pipeline_ok": PIPELINE_OK, "train_ok": TRAIN_OK, "model_tasks": MODEL_TASKS,
            "validation_run": VALIDATION_RUN, "validation_score": VALIDATION_SCORE, "facts": RUN_FACTS,
            "run_mode": RUN_MODE, "failures": FAILURES, "hours": hours_elapsed(), "num_gpus": NUM_GPUS}
    (WORKING / "submission_run_info.json").write_text(json.dumps(info, indent=2, default=str))
    print(json.dumps(info, indent=2, default=str), flush=True)
except Exception as error:
    print("Run-info write failed:", repr(error), flush=True)
SUBMISSION_VALID = False
try:
    competition_tasks = json.loads(Path(COMPETITION_TEST_FILE).read_text())
    try:
        _validate_submission(json.loads((WORKING / "submission.json").read_text()), competition_tasks)
        SUBMISSION_VALID = True
    except Exception as error:
        record_failure("final submission check", error)
        repaired = WORKING / "submission.json.tmp"
        repaired.write_text(json.dumps(identity_submission(competition_tasks)))
        repaired.replace(WORKING / "submission.json")
        print("submission.json replaced by an identity fallback for the competition tasks", flush=True)
except Exception as error:
    record_failure("final submission rescue", error)
relay()
# Failure telemetry (as v8). The identity file for the hidden tasks measured 1,073,534 bytes.
# When a submission run wrote no model answers, append `code` newlines so size - 1,073,534 = code:
#   code = (stage * 100 + gpus_trained * 10 + kind) * 1000 + minutes elapsed (capped at 999);
#   stage 1 environment, 2 runtime extraction, 3 checkpoint/challenges, 4 adaptation, 5 decode/pooling,
#   7 final submission check, 8 none recorded; kind of GPU 0's last attempt: 1 OOM, 2 ETA, 3 other, 0 none.
FAILURE_CODE = None
try:
    lost = any(f["stage"] == "final submission check" for f in FAILURES)
    if RUN_MODE == "submission" and (lost or (SUBMISSION_VALID and not MODEL_TASKS)):
        stages = {"environment": 1, "runtime extraction": 2, "checkpoint and challenges": 3,
                  "adaptation": 4, "pooled": 5, "gpu": 5, "final submission check": 7}
        stage = next((code for failure in FAILURES for prefix, code in stages.items()
                      if failure["stage"].startswith(prefix)), 8)
        trained = sum(bool(STATE[gpu]["train_ok"]) for gpu in range(NUM_GPUS)) if "STATE" in globals() else 0
        attempts = STATE[0]["attempts"] if "STATE" in globals() and STATE else []
        kind = {"oom": 1, "eta": 2, "other": 3}.get(attempts[-1]["result"], 0) if attempts else 0
        minutes = min(999, int(hours_elapsed() * 60))
        FAILURE_CODE = (stage * 100 + min(trained, 9) * 10 + kind) * 1000 + minutes
        padded = WORKING / "submission.json.tmp"
        padded.write_text(json.dumps(identity_submission(competition_tasks)) + "\n" * FAILURE_CODE)
        _validate_submission(json.loads(padded.read_text()), competition_tasks)
        padded.replace(WORKING / "submission.json")
        print(f"FAILURE CODE {FAILURE_CODE}: identity submission padded (stage {stage}, "
              f"gpus trained {trained}, kind {kind}, minute {minutes})", flush=True)
except Exception as error:
    print("Failure telemetry skipped:", repr(error), flush=True)
print(f"READY: mode {RUN_MODE}; submission.json valid for the competition tasks: {SUBMISSION_VALID}; "
      f"tasks with model answers: {MODEL_TASKS if RUN_MODE == 'submission' else 0}; "
      f"hours {hours_elapsed():.2f}", flush=True)
